In [1]:
# ================== 0. Конфиг ==================
USE_LEMMATIZE = True         # Лемматизация pymorphy2 (медленно, но иногда точнее)
DROP_ZERO_TARGET = True       # Удалить объекты с target == 0 (в задаче это "нет рейтинга")
N_FOLDS = 5
RANDOM_STATE = 42
CLIP_TO_1_5 = True            # Клиповать предсказания в [1, 5]
VERBOSE = 200

import os, re, zipfile, warnings
from collections import defaultdict
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error

import catboost as cb
import torch

np.random.seed(RANDOM_STATE)

# ================== 1. Загрузка и распаковка ==================
def extract_if_exists(path):
    if os.path.exists(path):
        with zipfile.ZipFile(path, 'r') as z:
            z.extractall(os.path.dirname(path))
        print(f"Распакован архив: {os.path.basename(path)}")
    else:
        print(f"Предупреждение: архив {os.path.basename(path)} не найден.")

base_dir = os.getcwd()
for f in ['train.tsv', 'test.tsv', 'reviews.txv']:
    p = os.path.join(base_dir, f)
    if os.path.isfile(p):
        try:
            os.remove(p)
            print(f"Удалён старый файл: {f}")
        except Exception as e:
            print(f"Не удалось удалить {f}: {e}")

# Попробуем несколько вариантов имён архивов отзывов (встречалась опечатка)
extract_if_exists(os.path.join(base_dir, 'train.tsv.zip'))
extract_if_exists(os.path.join(base_dir, 'test.tsv.zip'))
if os.path.exists(os.path.join(base_dir, 'reviews.txv.zip')):
    extract_if_exists(os.path.join(base_dir, 'reviews.txv.zip'))

# Читаем tsv/csv
def read_any(path_tsv, path_csv=None):
    if os.path.exists(path_tsv):
        return pd.read_csv(path_tsv, sep='\t')
    if path_csv and os.path.exists(path_csv):
        return pd.read_csv(path_csv)
    raise FileNotFoundError(f"Не найдено: {path_tsv} и {path_csv}")

train_df = read_any(os.path.join(base_dir, 'train.tsv'))
test_df  = read_any(os.path.join(base_dir, 'test.tsv'))
# отзывы могут быть в tsv или csv
try:
    reviews_df = read_any(os.path.join(base_dir, 'reviews.tsv'), os.path.join(base_dir, 'reviews.csv'))
except FileNotFoundError:
    # fallback: пустые отзывы
    warnings.warn("reviews.* не найден, продолжаю без текстов.")
    reviews_df = pd.DataFrame({'id': [], 'text': []})

print(f"train: {train_df.shape}, test: {test_df.shape}, reviews: {reviews_df.shape}")

# ================== 2. Агрегация отзывов и merge ==================
reviews_df['text'] = reviews_df['text'].fillna('').astype(str)
reviews_agg = reviews_df.groupby('id', as_index=False)['text'].apply(lambda s: ' '.join(s.values))
reviews_agg.rename(columns={'text': 'full_review_text'}, inplace=True)

train_full = train_df.merge(reviews_agg, on='id', how='left')
test_full  = test_df.merge(reviews_agg, on='id', how='left')

for df in (train_full, test_full):
    df['full_review_text'] = df['full_review_text'].fillna('')
    if 'category' in df.columns:
        df['category'] = df['category'].fillna('unknown').astype(str)

Удалён старый файл: train.tsv
Удалён старый файл: test.tsv
Распакован архив: train.tsv.zip
Распакован архив: test.tsv.zip
Распакован архив: reviews.txv.zip
train: (41105, 286), test: (9276, 285), reviews: (440082, 2)


In [2]:
train_full['name']= train_full['name'].map(lambda x: x.lower() if isinstance(x,str) else x)

In [3]:
test_full['name']= test_full['name'].map(lambda x: x.lower() if isinstance(x,str) else x)

In [4]:
duplicate_values_col1 = train_full['name'].value_counts()

In [5]:
values_appearing_more_than_once = duplicate_values_col1[duplicate_values_col1 > 1].index.tolist()

In [6]:
len(values_appearing_more_than_once)

3207

In [7]:
train_full.loc[train_full['name'].isin(values_appearing_more_than_once),'m_target'] = train_full[train_full['name'].isin(values_appearing_more_than_once)].groupby('name')['target'].transform('median')

In [8]:
#print(test_full)

In [9]:
namesandtargets = train_full[['name','m_target']].drop_duplicates()

test_full = test_full.merge(namesandtargets, on='name', how='left')
test_full.fillna(0, inplace=True)

In [10]:
#train_full['name'].isin(test_full['name'])

In [11]:
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, TrainingArguments, Trainer)

In [12]:
from sklearn.model_selection import train_test_split

In [13]:
train_full.drop(train_full[train_full.target < 1].index, inplace=True)

In [14]:
y=train_full["target"]

In [15]:
#y.value_counts()

In [16]:
MODEL_ID = "sergeyzh/berta"
MAX_LEN = 512

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def tokenize_texts(texts):
    return tokenizer(texts, truncation=True, padding=True, max_length=MAX_LEN)

# простой сплит
y=train_full["target"].tolist()
tr_texts, val_texts, tr_y, val_y = train_test_split(
    train_full["full_review_text"].tolist(),y , test_size=0.2, random_state=42, stratify=y
)

tr_enc = tokenize_texts(tr_texts)
val_enc = tokenize_texts(val_texts)


In [17]:
import torch

In [18]:
from torch.utils.data import Dataset
class Dataset12(torch.utils.data.Dataset):
    def __init__(self, encodings, labels=None):
        self.enc = encodings
        self.labels = labels
    def __len__(self):
        return len(self.enc["input_ids"])
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[i]).float()
        return item

train_ds = Dataset12(tr_enc, tr_y)
val_ds = Dataset12(val_enc, val_y)
collator = DataCollatorWithPadding(tokenizer)

In [19]:
modelbert = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=1,  trust_remote_code=True,
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at /home/roman/.cache/huggingface/hub/models--sergeyzh--rubert-mini-frida/snapshots/32a82388a9f99c99eca74494731f7ee6e9a9fe3a and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [20]:
import numpy as np
from sklearn.metrics import mean_squared_error

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    rmse = np.sqrt(mean_squared_error(labels, predictions))
    return {"rmse": rmse}
from sklearn.metrics import accuracy_score, f1_score

# def compute_metrics(pred):
#     labels = pred.label_ids
#     preds = pred.predictions.argmax(-1)
#     f1 = f1_score(labels, preds, average="weighted")
#     acc = accuracy_score(labels, preds)
#     return {"accuracy": acc, "f1": f1}

In [21]:
#del trainer

In [22]:
torch.cuda.empty_cache()
fp16_ok = torch.cuda.is_available() # Проверяем, доступен ли GPU для ускорения вычислений

args = TrainingArguments(
    output_dir="bert_baseline_2",
    per_device_train_batch_size=32, # Размер батча для обучения
    per_device_eval_batch_size=32, # Размер батча для валидации
    num_train_epochs=4, # Сколько раз "прогнать" все данные
    learning_rate=1e-5, # Скорость обучения. Стандартное значение для Бертов
    weight_decay=0.1, # Параметр регуляризации для предотвращения переобучения
    logging_steps=100,
    save_steps=100,
    do_eval=True,
    bf16=fp16_ok, # Использовать 16-битную точность для ускорения (если есть GPU)
    report_to="none",
    seed=42,
    torch_compile=True, # optimizations
    #optim="adamw_torch_fused", # improved optimizer

    logging_steps=100,
    save_steps=100,
    do_eval=True,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",  # Use F1-score for best model selection
    eval_strategy='steps',
    save_strategy='steps',
)

# Создаем объект Trainer, передавая ему все, что мы подготовили
trainer = Trainer(
    model=modelbert,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=collator, # Инструмент для создания батчей (с паддингом)
    compute_metrics=compute_metrics
)

# Запускаем обучение
trainer.train() #resume_from_checkpoint = True

# Опционально - можно сохранить модельку
# trainer.save_model("bert_baseline/best")
# tokenizer.save_pretrained("bert_baseline/best")

/home/roman/anaconda3/envs/testenv/lib/python3.11/site-packages/torch/backends/__init__.py:46: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  self.setter(val)


Step,Training Loss,Validation Loss,Rmse
100,10.575800,6.577804,2.564723
200,5.096900,2.903559,1.703983
300,2.153500,1.025921,1.012877
400,0.708600,0.260031,0.509932
500,0.233200,0.193671,0.440080
600,0.178900,0.173600,0.416654
700,0.172600,0.177864,0.421739
800,0.168800,0.162353,0.402931
900,0.171500,0.160046,0.400058
1000,0.162700,0.152434,0.390428


TrainOutput(global_step=3712, training_loss=0.636377887190159, metrics={'train_runtime': 754.0248, 'train_samples_per_second': 157.528, 'train_steps_per_second': 4.923, 'total_flos': 1995551800872960.0, 'train_loss': 0.636377887190159, 'epoch': 4.0})

In [23]:
test_text = test_full['full_review_text'].tolist()

test_enc = tokenize_texts(test_text)
test_ds = Dataset12(test_enc)

subm = trainer.predict(test_ds)

test_full['ptarget'] = subm.predictions.flatten()

# submission_df = pd.DataFrame({
#     'id': test_full['id'],
#     'target': subm.predictions.flatten()
# })


# submission_df['target'] = submission_df['target']

# print("\nПример submission файла:")
# try:
#     display(submission_df.head())
# except NameError:
#     print(submission_df.head())

# # Сохраняем submission в файл
# submission_df.to_csv('submission.csv', index=False)
# print("Submission файл 'submission.csv' успешно сохранен.")

In [24]:
#test_full['ptarget'] = submission_df['target']

In [25]:
train_text = train_full['full_review_text'].tolist()

train_enc = tokenize_texts(train_text)
train_ds = Dataset12(train_enc)

subm1 = trainer.predict(train_ds)
train_full['ptarget'] = subm1.predictions.flatten()


In [26]:
# ================== 3. (Опц.) лемматизация ==================
if USE_LEMMATIZE:
    # Патч для pymorphy2 на Python 3.12
    import inspect
    from collections import namedtuple
    ArgSpec = namedtuple('ArgSpec', 'args varargs keywords defaults')
    def _compat_getargspec(func):
        fspec = inspect.getfullargspec(func)
        return ArgSpec(fspec.args, fspec.varargs, fspec.varkw, fspec.defaults)
    inspect.getargspec = _compat_getargspec

    import nltk, re
    from nltk.corpus import stopwords
    try:
        russian_stopwords = set(stopwords.words('russian'))
    except LookupError:
        nltk.download('stopwords')
        russian_stopwords = set(stopwords.words('russian'))

    from pymorphy2 import MorphAnalyzer
    morph = MorphAnalyzer()
    from functools import lru_cache

    @lru_cache(maxsize=200_000)
    def norm_form(token: str) -> str:
        return morph.parse(token)[0].normal_form

    def preprocess_text(text: str) -> str:
        if not isinstance(text, str) or not text:
            return ""
        text = text.lower()
        text = re.sub(r'[^а-яё\s]', ' ', text)
        words = [w for w in text.split() if w not in russian_stopwords]
        return " ".join(norm_form(w) for w in words)

    tqdm.pandas(desc="Лемматизация")
    train_full['full_review_text'] = train_full['full_review_text'].progress_apply(preprocess_text)
    test_full['full_review_text']  = test_full['full_review_text'].progress_apply(preprocess_text)

    train_full['name'] = train_full['name'].progress_apply(preprocess_text)
    test_full['name']  = test_full['name'].progress_apply(preprocess_text)

# ================== 4. Фичи по координатам и текстовые статы ==================
def parse_coords_column(df: pd.DataFrame, col='coordinates'):
    if col not in df.columns:
        return df
    def parse_one(s):
        if pd.isna(s):
            return np.nan, np.nan
        s = str(s)
        nums = re.findall(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", s)
        if len(nums) >= 2:
            try:
                return float(nums[0]), float(nums[1])
            except:
                return np.nan, np.nan
        return np.nan, np.nan

    lon_lat = df[col].apply(parse_one)
    df['lon'] = [x[0] for x in lon_lat]
    df['lat'] = [x[1] for x in lon_lat]
    return df

def add_text_stats(df: pd.DataFrame, text_col='full_review_text'):
    s = df[text_col].fillna('').astype(str)
    df['review_len'] = s.str.len().astype(np.int32)
    df['review_words'] = s.str.split().str.len().fillna(0).astype(np.int32)
    df['review_sentences'] = s.str.count(r'[.!?]+').fillna(0).astype(np.int32)
    return df

def add_radius_ratios(df: pd.DataFrame):
    # Авто-фичи по парам *_300m и *_1000m: ratio и diff
    cols = list(df.columns)
    base_to_suffix = defaultdict(dict)
    for c in cols:
        if c.endswith('_300m'):
            base_to_suffix[c[:-5]]['300'] = c
        elif c.endswith('_1000m'):
            base_to_suffix[c[:-6]]['1000'] = c
    for base, d in base_to_suffix.items():
        if '300' in d and '1000' in d:
            a = d['300']
            b = d['1000']
            if np.issubdtype(df[a].dtype, np.number) and np.issubdtype(df[b].dtype, np.number):
                df[f'{base}_ratio_300_1000'] = (df[a] / (df[b].replace(0, np.nan))).fillna(0).astype(np.float32)
                df[f'{base}_diff_1000_300'] = (df[b] - df[a]).astype(np.float32)
    return df

for df in (train_full, test_full):
    df = parse_coords_column(df, 'coordinates')
    df = add_text_stats(df, 'full_review_text')
    df = add_radius_ratios(df)

# Частота категории (count-encoding как числовая фича)
if 'category' in train_full.columns:
    cat_counts = train_full['category'].value_counts().to_dict()
    train_full['category_count'] = train_full['category'].map(cat_counts).astype(np.int32)
    test_full['category_count']  = test_full['category'].map(cat_counts).fillna(0).astype(np.int32)

# ================== 5. Подготовка списков признаков ==================
DROP_COLS = ['id',  'address', 'coordinates', 'target']  # удаляем из X 'name',
TEXT_COLS = [c for c in ['name','full_review_text'] if c in train_full.columns]
CAT_COLS  = [c for c in ['category'] if c in train_full.columns]

Лемматизация:   0%|          | 0/37119 [00:00<?, ?it/s]

Лемматизация:   0%|          | 0/9276 [00:00<?, ?it/s]

Лемматизация:   0%|          | 0/37119 [00:00<?, ?it/s]

Лемматизация:   0%|          | 0/9276 [00:00<?, ?it/s]

/tmp/ipykernel_1397412/4241436728.py:85: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{base}_ratio_300_1000'] = (df[a] / (df[b].replace(0, np.nan))).fillna(0).astype(np.float32)
/tmp/ipykernel_1397412/4241436728.py:86: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{base}_diff_1000_300'] = (df[b] - df[a]).astype(np.float32)
/tmp/ipykernel_1397412/4241436728.py:85: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider j

In [27]:
print(test_full['m_target'])

0       4.2
1       3.4
2       0.0
3       3.5
4       0.0
       ... 
9271    0.0
9272    3.9
9273    0.0
9274    3.6
9275    3.8
Name: m_target, Length: 9276, dtype: float64


In [28]:
train_full['full_review_text']

0        добрый вечер отличный поликлиника смысл особен...
1        отличный аутентичный место высокий точка столи...
2        огромный количество сотрудник несколько больни...
3        ходить салон регулярно особенно нравиться маст...
4        отличный организация персонал приветливый вежл...
                               ...                        
41099    ранее место находиться начальный ступень й шко...
41100    отлично вкусно быстро доступно отличный локаль...
41101    всё пройти отлично первый заходить сюда особен...
41103    зайти отзыв выбрать букет мама действительно п...
41104    вгик один хороший кинематографический школа ху...
Name: full_review_text, Length: 37119, dtype: object

In [30]:
del trainer
torch.cuda.empty_cache()

In [32]:
train_full.columns

Index(['id', 'name', 'coordinates', 'category', 'address', 'target',
       'traffic_300m', 'homes_300m', 'works_300m', 'female_300m',
       ...
       'goods_for_moms_and_babies_diff_1000_300', 'age_25-34_ratio_300_1000',
       'age_25-34_diff_1000_300', 'male_ratio_300_1000', 'male_diff_1000_300',
       'phone_repair_ratio_300_1000', 'phone_repair_diff_1000_300',
       'mean_income_ratio_300_1000', 'mean_income_diff_1000_300',
       'category_count'],
      dtype='object', length=575)

In [36]:
def build_feature_matrix(df: pd.DataFrame, feature_cols):
    X = df[feature_cols].copy()
    # Привести типы: текст оставить строкой, остальное в числа
    for c in X.columns:
        if c in CAT_COLS or c in TEXT_COLS:
            X[c] = X[c].astype(str)
        else:
            if not np.issubdtype(X[c].dtype, np.number):
                X[c] = pd.to_numeric(X[c], errors='coerce')
            X[c] = X[c].fillna(0)
            if X[c].dtype != np.float32:
                X[c] = X[c].astype(np.float32)
    return X

# целевая
assert 'target' in train_full.columns, "В train нет target."
y = train_full['target'].astype(float)

# по желанию — убрать "0" как отсутствие рейтинга
if DROP_ZERO_TARGET:
    mask = y > 0
    dropped = (~mask).sum()
    if dropped > 0:
        print(f"Удаляю {dropped} объектов с target == 0")
    train_full = train_full[mask].reset_index(drop=True)
    y = y[mask].reset_index(drop=True)

# Список фич: все, кроме DROP_COLS
feature_cols = [c for c in train_full.columns if c not in DROP_COLS]
# Убедимся, что текст и категория в списке
feature_cols = [c for c in feature_cols if c in train_full.columns]

X_all = build_feature_matrix(train_full, feature_cols)
X_test_all = build_feature_matrix(test_full, feature_cols)

# Индексы/имена кат и текст фич
cat_idx  = [feature_cols.index(c) for c in CAT_COLS if c in feature_cols]
text_idx = [feature_cols.index(c) for c in TEXT_COLS if c in feature_cols]

print("Число признаков:", len(feature_cols))
print("Категориальные:", CAT_COLS)
print("Текстовые:", TEXT_COLS)

# ================== 6. CV: оценка и подбор итераций ==================
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
oof = np.zeros(len(X_all), dtype=float)
best_iters = []

for fold, (tr, vl) in enumerate(kf.split(X_all), 1):
    print(f"\n=== Fold {fold}/{N_FOLDS} ===")
    train_pool = cb.Pool(
        X_all.iloc[tr], label=y.iloc[tr],
        cat_features=cat_idx if cat_idx else None,
       text_features=text_idx if text_idx else None
    )
    val_pool = cb.Pool(
        X_all.iloc[vl], label=y.iloc[vl],
        cat_features=cat_idx if cat_idx else None,
        text_features=text_idx if text_idx else None
    )

    model = cb.CatBoostRegressor(
        iterations=4000,
        learning_rate=0.03,
        depth=10,
        l2_leaf_reg=15,
        loss_function='MAE',
        eval_metric='MAE',
        random_seed=RANDOM_STATE,
        verbose=VERBOSE,
        early_stopping_rounds=300,
        task_type="GPU" if torch.cuda.is_available() else "CPU",
        # при желании: feature_calcers=["BoW", "BM25", "NaiveBayes"]
    )
    model.fit(train_pool, eval_set=val_pool, use_best_model=True)
    pred = model.predict(val_pool)
    oof[vl] = pred
    fold_mae = mean_absolute_error(y.iloc[vl], pred)
    best_iters.append(model.get_best_iteration() or model.tree_count_)
    print(f"Fold MAE: {fold_mae:.5f}, best_iter: {best_iters[-1]}")

cv_mae = mean_absolute_error(y, oof)
print(f"\nOOF MAE: {cv_mae:.5f}")
suggested_iters = int(np.median(best_iters) * 1.1)
print(f"Рекомендуемое число итераций для финального обучения: {suggested_iters}")


Число признаков: 571
Категориальные: ['category']
Текстовые: ['name', 'full_review_text']

=== Fold 1/5 ===


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 0.3249306	test: 0.3213061	best: 0.3213061 (0)	total: 346ms	remaining: 23m 2s
200:	learn: 0.2213731	test: 0.2199778	best: 0.2199778 (200)	total: 39.6s	remaining: 12m 29s
400:	learn: 0.2137085	test: 0.2178348	best: 0.2178098 (397)	total: 1m 18s	remaining: 11m 43s
600:	learn: 0.2075318	test: 0.2172229	best: 0.2172151 (576)	total: 1m 57s	remaining: 11m 5s
800:	learn: 0.2011817	test: 0.2169136	best: 0.2168993 (789)	total: 2m 38s	remaining: 10m 31s
1000:	learn: 0.1954299	test: 0.2166278	best: 0.2166234 (988)	total: 3m 17s	remaining: 9m 52s
1200:	learn: 0.1898464	test: 0.2165069	best: 0.2164727 (1181)	total: 3m 58s	remaining: 9m 15s
1400:	learn: 0.1842506	test: 0.2163995	best: 0.2163865 (1386)	total: 4m 38s	remaining: 8m 37s
1600:	learn: 0.1789160	test: 0.2162786	best: 0.2162194 (1583)	total: 5m 19s	remaining: 7m 58s
1800:	learn: 0.1737962	test: 0.2162900	best: 0.2162194 (1583)	total: 6m	remaining: 7m 19s
bestTest = 0.2162193759
bestIteration = 1583
Shrink model to first 1584 iterat

Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 0.3228443	test: 0.3263423	best: 0.3263423 (0)	total: 208ms	remaining: 13m 52s
200:	learn: 0.2196920	test: 0.2298107	best: 0.2298107 (200)	total: 39.1s	remaining: 12m 18s
400:	learn: 0.2120002	test: 0.2273662	best: 0.2273654 (399)	total: 1m 19s	remaining: 11m 49s
600:	learn: 0.2049203	test: 0.2263777	best: 0.2263777 (600)	total: 1m 58s	remaining: 11m 12s
800:	learn: 0.1987724	test: 0.2257569	best: 0.2257569 (800)	total: 2m 37s	remaining: 10m 31s
1000:	learn: 0.1929964	test: 0.2253237	best: 0.2252946 (992)	total: 3m 18s	remaining: 9m 55s
1200:	learn: 0.1873398	test: 0.2250651	best: 0.2250299 (1180)	total: 3m 59s	remaining: 9m 17s
1400:	learn: 0.1819981	test: 0.2249964	best: 0.2249519 (1326)	total: 4m 38s	remaining: 8m 37s
1600:	learn: 0.1763831	test: 0.2247831	best: 0.2247736 (1534)	total: 5m 19s	remaining: 7m 58s
1800:	learn: 0.1712272	test: 0.2244974	best: 0.2244859 (1796)	total: 6m	remaining: 7m 20s
2000:	learn: 0.1662432	test: 0.2243553	best: 0.2243135 (1991)	total: 6m 45s	

...

0:	learn: 0.3240147	test: 0.3212639	best: 0.3212639 (0)	total: 197ms	remaining: 13m 8s
200:	learn: 0.2208486	test: 0.2229098	best: 0.2229098 (200)	total: 38.7s	remaining: 12m 11s
400:	learn: 0.2137329	test: 0.2209802	best: 0.2209411 (393)	total: 1m 17s	remaining: 11m 39s
600:	learn: 0.2075907	test: 0.2205104	best: 0.2205104 (600)	total: 1m 57s	remaining: 11m 3s
800:	learn: 0.2014874	test: 0.2201857	best: 0.2201803 (799)	total: 2m 36s	remaining: 10m 24s
1000:	learn: 0.1957591	test: 0.2199678	best: 0.2199336 (969)	total: 3m 17s	remaining: 9m 50s
1200:	learn: 0.1902192	test: 0.2198058	best: 0.2198040 (1190)	total: 3m 57s	remaining: 9m 13s
1400:	learn: 0.1846687	test: 0.2196896	best: 0.2196738 (1397)	total: 4m 37s	remaining: 8m 34s
1600:	learn: 0.1795823	test: 0.2196429	best: 0.2195660 (1448)	total: 5m 18s	remaining: 7m 56s
1800:	learn: 0.1744295	test: 0.2196039	best: 0.2195138 (1775)	total: 5m 58s	remaining: 7m 18s
2000:	learn: 0.1695659	test: 0.2195693	best: 0.2195138 (1775)	total: 6m 39

Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 0.3223920	test: 0.3267750	best: 0.3267750 (0)	total: 213ms	remaining: 14m 10s
200:	learn: 0.2203721	test: 0.2283719	best: 0.2283719 (200)	total: 40s	remaining: 12m 35s
400:	learn: 0.2133946	test: 0.2258263	best: 0.2258263 (400)	total: 1m 19s	remaining: 11m 49s
600:	learn: 0.2065373	test: 0.2245180	best: 0.2245180 (600)	total: 1m 58s	remaining: 11m 12s
800:	learn: 0.2005093	test: 0.2238697	best: 0.2238697 (800)	total: 2m 39s	remaining: 10m 35s
1000:	learn: 0.1944683	test: 0.2234054	best: 0.2233845 (990)	total: 3m 19s	remaining: 9m 57s
1200:	learn: 0.1886461	test: 0.2230002	best: 0.2229841 (1197)	total: 3m 59s	remaining: 9m 18s
1400:	learn: 0.1831657	test: 0.2229853	best: 0.2229274 (1289)	total: 4m 39s	remaining: 8m 39s
1600:	learn: 0.1775984	test: 0.2228921	best: 0.2228856 (1599)	total: 5m 21s	remaining: 8m 1s
1800:	learn: 0.1724422	test: 0.2227269	best: 0.2226922 (1775)	total: 6m 1s	remaining: 7m 21s
2000:	learn: 0.1672129	test: 0.2228270	best: 0.2226922 (1775)	total: 6m 43s	

Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 0.3228526	test: 0.3257327	best: 0.3257327 (0)	total: 205ms	remaining: 13m 39s
200:	learn: 0.2201794	test: 0.2251972	best: 0.2251972 (200)	total: 39.9s	remaining: 12m 34s
400:	learn: 0.2130427	test: 0.2233202	best: 0.2233168 (397)	total: 1m 18s	remaining: 11m 44s
600:	learn: 0.2062308	test: 0.2224839	best: 0.2224839 (600)	total: 1m 58s	remaining: 11m 10s
800:	learn: 0.1994836	test: 0.2217938	best: 0.2217573 (790)	total: 2m 39s	remaining: 10m 35s
1000:	learn: 0.1933139	test: 0.2214064	best: 0.2214002 (998)	total: 3m 18s	remaining: 9m 54s
1200:	learn: 0.1873884	test: 0.2212712	best: 0.2212712 (1200)	total: 3m 59s	remaining: 9m 17s
1400:	learn: 0.1816181	test: 0.2207779	best: 0.2207749 (1397)	total: 4m 39s	remaining: 8m 38s
1600:	learn: 0.1763977	test: 0.2206730	best: 0.2206111 (1509)	total: 5m 20s	remaining: 7m 59s
1800:	learn: 0.1713882	test: 0.2205025	best: 0.2204931 (1792)	total: 6m 1s	remaining: 7m 21s
2000:	learn: 0.1663051	test: 0.2204535	best: 0.2204074 (1985)	total: 6m 4

In [37]:

# ================== 7. Финальное обучение на всех данных ==================
full_pool = cb.Pool(
    X_all, label=y,
    cat_features=cat_idx if cat_idx else None,
    text_features=text_idx if text_idx else None
)
final_model = cb.CatBoostRegressor(
    #iterations=max(suggested_iters, 800),
    iterations = suggested_iters,
    learning_rate=0.03,
    #learning_rate=0.05,
    depth=10,
    l2_leaf_reg=15,
    loss_function='MAE',
    eval_metric='MAE',
    random_seed=RANDOM_STATE,
    verbose=VERBOSE,
    # Для финала отключаем раннюю остановку (нет валидации):
    early_stopping_rounds=None,
    task_type="GPU" if torch.cuda.is_available() else "CPU",
)
final_model.fit(full_pool, verbose=VERBOSE)


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 0.3232485	total: 209ms	remaining: 7m 36s
200:	learn: 0.2202928	total: 39.5s	remaining: 6m 29s
400:	learn: 0.2143191	total: 1m 18s	remaining: 5m 50s
600:	learn: 0.2081165	total: 1m 59s	remaining: 5m 13s
800:	learn: 0.2027440	total: 2m 38s	remaining: 4m 32s
1000:	learn: 0.1976873	total: 3m 18s	remaining: 3m 54s
1200:	learn: 0.1925224	total: 4m	remaining: 3m 16s
1400:	learn: 0.1879785	total: 4m 39s	remaining: 2m 36s
1600:	learn: 0.1834342	total: 5m 20s	remaining: 1m 56s
1800:	learn: 0.1788610	total: 6m 1s	remaining: 1m 16s
2000:	learn: 0.1746342	total: 6m 41s	remaining: 36.5s
2182:	learn: 0.1708787	total: 7m 19s	remaining: 0us


In [38]:
# ================== 8. Предсказание и сабмит ==================
test_pool = cb.Pool(
    X_test_all,
    cat_features=cat_idx if cat_idx else None,
    text_features=text_idx if text_idx else None
)
test_pred = final_model.predict(test_pool)

#if CLIP_TO_1_5:
#    test_pred = np.clip(test_pred, 1.0, 5.0)

submission = pd.DataFrame({
    'id': test_full['id'].values,
    'target': test_pred
})
display(submission.head())
submission.to_csv('submission.csv', index=False)
print("submission.csv сохранён.")

,id,target
0,21472,4.025754
1,9837,3.181784
2,41791,4.122441
3,18441,3.417741
4,49348,3.323884


submission.csv сохранён.
